# Phá hoại tài sản · probe_02: dữ liệu âm khó + fine-tune vài block cuối V-JEPA 2

Notebook này train lại extension phá hoại tài sản để giảm báo nhầm với các động tác **vung tay và đập vào vật hợp lệ**
(chẻ củi, đập ván, chặt thịt, cưa gỗ, đào đất, vung gậy bóng chày, ném rìu, đấm bao cát, đá cao, đá ngang…).
Bản probe_01 báo nhầm 8/12 clip chẻ củi.

**Cách chạy:**
1. Vào *Runtime → Change runtime type*, chọn **A100** (L4 cũng chạy được nhưng chậm hơn).
2. Vào *Runtime → Run all*. Đến bước mount Google Drive thì cho phép truy cập.
3. Không cần ngồi chờ. Phiên Colab **tự ngắt** khi:
   - chạy xong mọi bước, hoặc
   - gặp lỗi (traceback được ghi ra `ERROR.txt` trên Drive)

   nên không tốn compute unit vô ích.

**Các bước:**

| Bước | Việc làm | Thời gian ước tính (A100) |
|---|---|---|
| 1 | Tải dữ liệu: như probe_01, thêm 11 lớp âm khó × 400 clip, trọng số loss ×2 | 40–70 phút |
| 2 | Trích feature V-JEPA 2 đóng băng | 60–120 phút |
| 3 | **Bản A – frozen:** train lại attentive probe trên dữ liệu mới. Ngưỡng chọn trên dev: precision ≥ 0.9, mỗi lớp âm khó báo nhầm ≤ 10% | ~10 phút |
| 4 | Lưu trạng thái ẩn đi vào 4 block cuối cho 3 cửa sổ "giống phá hoại nhất" của mỗi clip train/dev | 45–90 phút |
| 5 | **Bản B – fine-tune:** train 4 block cuối (của 24) + LayerNorm cuối + probe. Khởi đầu từ bản A, lr nhỏ, giảm lr theo tầng, dừng sớm theo AP trên dev | 30–90 phút |
| 6 | Mã hoá lại dev + test bằng encoder đã fine-tune (đúng đường chạy runtime), chọn ngưỡng cùng quy tắc với bản A, rồi in bảng so sánh A và B | 20–40 phút |

**Kết quả trên Drive:** `MyDrive/SENTINEL/artifacts/vandalism/<RUN_TAG>/`
- `frozen/vandalism_<RUN_TAG>_frozen.zip` (bản A)
- `finetune/vandalism_<RUN_TAG>_ft.zip` (bản B, có thêm `vandalism_tail.pt` khoảng 100 MB)
- `finetune/comparison.json`
- log của từng bước

Runtime chạy bản B tốn thời gian như bản A, vì cùng kiến trúc, chỉ khác trọng số.
Bản nào được cài sẽ quyết định sau khi đo trên máy, gồm video test thật và 13 video mẫu.

Ổ đĩa phiên cần khoảng 25 GB cho dữ liệu và tối đa khoảng 65 GB cho cache ở bước 4.
Nếu ổ không đủ chỗ, script tự giảm xuống 2 hoặc 1 cửa sổ mỗi clip.

Giấy phép nguồn: Kinetics (nhãn CC BY 4.0, video thuộc chủ YouTube), ASHEN (CC BY-NC 4.0), Charades (phi thương mại).
Chỉ dùng cho đồ án / nghiên cứu.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, traceback, urllib.request, zipfile
from google.colab import drive

RUN_TAG = 'vjepa2_probe_02'   # đổi tên khi chạy thí nghiệm mới
TAIL_BLOCKS = 4               # số block cuối (của 24) được fine-tune
FOCUS_PER_CLASS = 400         # clip mỗi lớp âm khó (vung tay đập vật hợp lệ)
AUTO_DISCONNECT = True        # tự ngắt phiên khi xong hoặc khi lỗi
CACHE_ON_DRIVE = False        # True: giữ feature đóng băng trên Drive (tốn 5–15 GB)
SOURCE_REVISION = '79e8f5a4400bbae5c7357c45094728d4d9578933'

assert RUN_TAG and all(c.isalnum() or c in '_-' for c in RUN_TAG)
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive', timeout_ms=180000)
roots = [p for p in (Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')) if p.is_dir()]
assert roots, 'Không thấy MyDrive sau khi mount Google Drive'
OUT = roots[0] / 'SENTINEL' / 'artifacts' / 'vandalism' / RUN_TAG
FROZEN, TUNED = OUT / 'frozen', OUT / 'finetune'
for path in (OUT, FROZEN, TUNED):
    path.mkdir(parents=True, exist_ok=True)
(OUT / 'drive_write_probe.txt').write_text('write OK', encoding='utf-8')
WORK = Path('/content/vandalism_work')
SRC, DATA = WORK / 'src', WORK / 'data'
CACHE = (OUT / 'feature_cache') if CACHE_ON_DRIVE else (WORK / 'cache')
TAIL_CACHE, EVAL_CACHE = WORK / f'tail_cache_{TAIL_BLOCKS}', WORK / 'eval_cache_ft'
for path in (WORK, DATA, CACHE):
    path.mkdir(parents=True, exist_ok=True)
STARTED = time.time()


def disconnect(reason):
    print(reason, flush=True)
    if AUTO_DISCONNECT:
        print('Phiên Colab sẽ tự ngắt sau 20 giây (đặt AUTO_DISCONNECT = False để giữ lại).', flush=True)
        time.sleep(20)  # để Drive kịp ghi xong
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()


def on_error(shell, etype, evalue, tb, tb_offset=None):
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    text = ''.join(traceback.format_exception(etype, evalue, tb))
    (OUT / 'ERROR.txt').write_text(time.strftime('%Y-%m-%d %H:%M:%S ') + text, encoding='utf-8')
    disconnect(f'LỖI, xem {OUT / "ERROR.txt"} và log trên Drive.')


get_ipython().set_custom_exc((Exception,), on_error)  # lỗi ở bất kỳ cell nào -> ghi Drive + tự ngắt
(OUT / 'ERROR.txt').unlink(missing_ok=True)
print('Kết quả sẽ lưu tại:', OUT)

In [ ]:
# Mã nguồn được ghim theo commit để kết quả tái lập được.
marker = SRC / '.revision'
if not marker.is_file() or marker.read_text().strip() != SOURCE_REVISION:
    shutil.rmtree(SRC, ignore_errors=True)
    archive = WORK / 'source.zip'
    url = 'https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/' + SOURCE_REVISION
    with urllib.request.urlopen(url, timeout=300) as response, archive.open('wb') as output:
        shutil.copyfileobj(response, output)
    with zipfile.ZipFile(archive) as package:
        top = package.namelist()[0].split('/')[0]
        package.extractall(WORK)
    (WORK / top).rename(SRC)
    marker.write_text(SOURCE_REVISION)
print('Mã nguồn:', SOURCE_REVISION)


def run(args, log_name, cwd=SRC):
    """Chạy lệnh, hiện log trực tiếp và lưu log ra Drive."""
    started = time.time()
    with (OUT / log_name).open('a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(map(str, args))}\n')
        process = subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        code = process.wait()
    print(f'--- {log_name}: exit {code}, {(time.time() - started) / 60:.1f} phút')
    if code:
        raise RuntimeError(f'{log_name} lỗi (exit {code}), xem {OUT / log_name}')


def py(module, *args, log):
    run([sys.executable, '-u', '-m', module, *args], log)

In [ ]:
# Thư viện: torch có sẵn trên Colab; ghim transformers cho V-JEPA 2.
run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.1'], 'install.log')
import torch, transformers
assert torch.cuda.is_available(), 'Chọn GPU (A100/L4) trong Runtime → Change runtime type'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| transformers', transformers.__version__)
print('Ổ đĩa trống: %.0f GB' % (shutil.disk_usage('/content').free / 1e9))

## Bước 1 · Tải dữ liệu và tạo manifest

Giống probe_01 (dương tính Kinetics `smashing` + `breaking glass` + ASHEN, âm khó Kinetics + Charades + ASHEN normal),
thêm **11 lớp âm khó** ở mức 400 clip mỗi lớp, trọng số loss ×2, và 30 clip test mỗi lớp.
Chia train/dev/test không trùng video gốc.

In [ ]:
py('extensions.vandalism.prepare_vandalism', '--root', DATA,
   '--neg-per-class', 150, '--neg-test-per-class', 30,
   '--focus-per-class', FOCUS_PER_CLASS, '--focus-weight', 2.0,
   '--ashen-normal-train', 60, '--ashen-normal-test', 60,
   '--charades-other-train', 1500, '--charades-other-test', 200, log='prepare.log')
shutil.copy(DATA / 'prepare_summary.json', OUT / 'prepare_summary.json')
print(json.dumps(json.loads((DATA / 'prepare_summary.json').read_text())['counts'], indent=1))
print('Ổ đĩa trống: %.0f GB' % (shutil.disk_usage('/content').free / 1e9))

## Bước 2 · Trích feature V-JEPA 2 đóng băng

Mỗi cửa sổ 8 khung (3.75 fps, khoảng 2.1 s, giống đầu vào runtime) cho ra 64 token × 1024.

In [ ]:
py('extensions.vandalism.train_vandalism', '--stage', 'features', '--data', DATA, '--cache', CACHE,
   '--batch', 32, log='features.log')

## Bước 3 · Bản A: attentive probe trên encoder đóng băng

Ngưỡng chọn trên dev: recall cao nhất mà precision ≥ 0.9, sau đó nâng lên cho tới khi
**mỗi lớp âm khó báo nhầm ≤ 10%** số clip dev của lớp đó.

In [ ]:
py('extensions.vandalism.train_vandalism', '--stage', 'train', '--data', DATA, '--cache', CACHE,
   '--output', FROZEN, '--run-tag', RUN_TAG + '_frozen', '--max-focus-false-alarm', 0.1, log='train_frozen.log')

## Bước 4 · Cache đầu vào của các block cuối

Mỗi clip train/dev lấy tối đa 3 cửa sổ có điểm "phá hoại" cao nhất theo bản A.
Đây chính là các cửa sổ mà pooling MIL top-3 dùng để học.
Các cửa sổ này được chạy qua 20 block đầu (đóng băng), rồi lưu trạng thái ẩn dạng fp16.
Nhờ vậy mỗi epoch fine-tune chỉ phải chạy 4 block cuối.

In [ ]:
py('extensions.vandalism.finetune_vandalism', '--stage', 'cache', '--data', DATA, '--cache', CACHE,
   '--frozen', FROZEN, '--tail-cache', TAIL_CACHE, '--tail-blocks', TAIL_BLOCKS, log='tail_cache.log')
print('Ổ đĩa trống: %.0f GB' % (shutil.disk_usage('/content').free / 1e9))

## Bước 5 · Bản B: fine-tune 4 block cuối + probe

Khởi đầu từ encoder gốc và probe của bản A. Epoch 0 chính là bản A.
- lr của block cuối là 2e-5, mỗi block thấp hơn có lr nhân thêm 0.75; lr của probe là 1e-4.
- Dùng bf16 và gradient checkpointing.
- Dừng sớm sau 3 epoch không cải thiện AP trên dev.
- Trọng số tốt nhất được ghi thẳng ra Drive: `finetune/tail_best.pt`.

In [ ]:
batch_bags = 8 if torch.cuda.get_device_properties(0).total_memory > 30e9 else 4
py('extensions.vandalism.finetune_vandalism', '--stage', 'finetune', '--data', DATA, '--frozen', FROZEN,
   '--tail-cache', TAIL_CACHE, '--output', TUNED, '--tail-blocks', TAIL_BLOCKS, '--batch-bags', batch_bags,
   '--epochs', 8, '--patience', 3, log='finetune.log')

## Bước 6 · Đánh giá bản B trên toàn bộ cửa sổ dev/test và so sánh với bản A

Dev và test được mã hoá lại bằng encoder đã fine-tune, giống hệt đường chạy runtime.
Ngưỡng được chọn bằng cùng quy tắc với bản A, nên so sánh hai bản là công bằng.

In [ ]:
py('extensions.vandalism.finetune_vandalism', '--stage', 'evaluate', '--data', DATA, '--frozen', FROZEN,
   '--output', TUNED, '--eval-cache', EVAL_CACHE, '--tail-blocks', TAIL_BLOCKS, '--batch', 32,
   '--run-tag', RUN_TAG + '_ft', '--max-focus-false-alarm', 0.1, log='evaluate_ft.log')

In [ ]:
comparison = json.loads((TUNED / 'comparison.json').read_text())
history = json.loads((TUNED / 'finetune_history.json').read_text())
print('Lịch sử fine-tune (epoch 0 = bản A):')
for row in history:
    print('  epoch', row['epoch'], '| loss', row['loss'], '| dev proxy AP', round(row['dev_proxy_ap'], 4))
print()
for name, row in comparison.items():
    print(name.upper(), {k: (round(v, 3) if isinstance(v, float) else v) for k, v in row.items() if k != 'false_alarm_by_class'})
print('\nBáo nhầm trên test theo lớp âm (A → B):')
for label in sorted(comparison['frozen']['false_alarm_by_class']):
    a = comparison['frozen']['false_alarm_by_class'].get(label)
    b = comparison['finetune']['false_alarm_by_class'].get(label)
    print(f'  {label:32s} {a!s:>6} → {b!s:>6}')
print('\nFile trên Drive:')
for path in sorted(OUT.rglob('*')):
    if path.is_file() and 'feature_cache' not in path.parts:
        print(f'  {str(path.relative_to(OUT)):48s} {path.stat().st_size / 1e6:8.2f} MB')
(OUT / 'DONE.txt').write_text(f'xong {time.strftime("%Y-%m-%d %H:%M:%S")}, {(time.time() - STARTED) / 3600:.1f} giờ',
                              encoding='utf-8')
print('\nTổng thời gian: %.1f giờ' % ((time.time() - STARTED) / 3600))
print('Bước tiếp theo: tải 2 file zip (frozen + finetune) về máy để đo và cài.')

## Tự ngắt phiên

Cell cuối ngắt phiên Colab để không tốn compute unit, vì kết quả đã nằm trên Drive.

In [ ]:
disconnect('XONG. Kết quả ở ' + str(OUT))